In [0]:
%sql
-- Amostragem inicial dos dados ingeridos
SELECT * 
FROM raw_ecommerce_enderecos 
LIMIT 100;

In [0]:
%sql
-- Análise de completude e contagem de campos nulos/vazios
SELECT 
    COUNT(*) AS total_linhas,
    COUNT(id_endereco) AS qtd_id_endereco,
    SUM(CASE WHEN logradouro IS NULL THEN 1 ELSE 0 END) AS nulos_logradouro,
    SUM(CASE WHEN numero IS NULL THEN 1 ELSE 0 END) AS nulos_numero,
    SUM(CASE WHEN complemento IS NULL THEN 1 ELSE 0 END) AS nulos_complemento,
    SUM(CASE WHEN bairro IS NULL THEN 1 ELSE 0 END) AS nulos_bairro,
    SUM(CASE WHEN cep IS NULL THEN 1 ELSE 0 END) AS nulos_cep,
    SUM(CASE WHEN cidade IS NULL THEN 1 ELSE 0 END) AS nulos_cidade,
    SUM(CASE WHEN estado IS NULL THEN 1 ELSE 0 END) AS nulos_estado,
    SUM(CASE WHEN latitude IS NULL OR longitude IS NULL THEN 1 ELSE 0 END) AS sem_coordenadas
FROM raw_ecommerce_enderecos;

In [0]:
%sql
-- Verificação de unicidade: Chaves primárias duplicadas
SELECT 
    id_endereco, 
    COUNT(*) AS qtd
FROM raw_ecommerce_enderecos
GROUP BY id_endereco
HAVING COUNT(*) > 1
ORDER BY qtd DESC;

In [0]:
%sql
-- Consistência de regras de negócio: Tamanho de CEP padronizado (apenas números)
SELECT 
    LENGTH(REGEXP_REPLACE(cep, '[^0-9]', '')) AS tamanho_cep,
    COUNT(*) AS total
FROM raw_ecommerce_enderecos
GROUP BY 1
ORDER BY total DESC;

In [0]:
%sql
-- Linhagem e auditoria: Lotes ingeridos por arquivo fonte
SELECT 
    source_file,
    MIN(ingestion_time) AS primeira_ingestao,
    MAX(ingestion_time) AS ultima_ingestao,
    COUNT(*) AS total_registros
FROM raw_ecommerce_enderecos
GROUP BY source_file
ORDER BY primeira_ingestao ASC;

In [0]:
### Célula 5 - análise exploratória

total = df.count()
print("Linhas:", total)

display(df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns if c != "arquivo_origem"]))
display(df.groupBy("id_endereco").count().filter("count > 1"))
display(df.filter("is_principal").groupBy("id_cliente").count().filter("count > 1"))
display(df.groupBy("estado").count().orderBy("estado"))
display(df.select("cep", F.length("cep").alias("tam_cep")).groupBy("tam_cep").count())
display(df.drop("arquivo_origem"))